# roadtraffic — end-to-end demo on real OSM roads

This notebook walks the **whole trafficability pipeline** on a real road network
pulled live from **OpenStreetMap (Overpass)**, using **GPS tracks that carry no
speed column** so it also showcases deriving speed from positions:

1. Download a small road network from Overpass (falls back to a synthetic grid
   if the API is unreachable — the notebook always runs).
2. Simulate timestamped GPS trajectories with unique vehicle ids and **no speed**.
3. **Derive speed** from successive positions and save the enriched dataset.
4. Match points to roads (nearest + HMM) and clean trajectories *dwell-aware*.
5. Aggregate by hour, detect **peak / off-peak** blocks.
6. Assign **three speeds per segment** (overall / peak / off-peak).
7. Route by time on each regime, with actionable errors.
8. Visualise segment speeds and a route on an interactive map.

> Extra (non-core) packages are used only for fetching and plotting:
> `requests`, `matplotlib`, `folium`. The `roadtraffic` core needs none of them.

## 0. Setup

Install the notebook-only extras and import. `roadtraffic` itself should already be installed (`pip install -e .` from the repo root).

In [ ]:
%pip install -q requests matplotlib folium

import json
import os
import tempfile
import warnings

import numpy as np
import pandas as pd
import requests
from shapely.geometry import LineString

import roadtraffic as rt

WORK = tempfile.mkdtemp(prefix="roadtraffic_demo_")
print("roadtraffic", rt.__version__, "| scratch dir:", WORK)

## 1. Download a road network from Overpass

We ask Overpass for drivable ways in a small bounding box and convert them to a
GeoJSON `LineString` FeatureCollection — exactly what `rt.Network.from_geojson`
expects. If every mirror is unreachable we fall back to a synthetic grid so the
rest of the notebook still runs.

Change `BBOX` to study your own area (keep it small — a few km² — to stay polite
to the public API).

In [ ]:
# (S, W, N, E) — a compact grid neighbourhood (Davis, CA core)
BBOX = (38.5430, -121.7450, 38.5480, -121.7380)
HIGHWAYS = "primary|secondary|tertiary|residential|unclassified|living_street"
MIRRORS = ("https://overpass-api.de/api/interpreter",
           "https://overpass.kumi.systems/api/interpreter")


def download_network_geojson(bbox, path):
    s, w, n, e = bbox
    q = (f'[out:json][timeout:60];way["highway"~"^({HIGHWAYS})$"]'
         f'({s},{w},{n},{e});out geom;')
    ua = {"User-Agent": "roadtraffic-example/0.1 (demo notebook)"}
    for url in MIRRORS:
        try:
            r = requests.post(url, data=q.encode(), headers=ua, timeout=60)
            if r.status_code != 200:
                continue
            elements = r.json().get("elements", [])
        except Exception:
            continue
        feats = []
        for el in elements:
            if el.get("type") != "way" or not el.get("geometry"):
                continue
            coords = [[p["lon"], p["lat"]] for p in el["geometry"]]
            if len(coords) < 2:
                continue
            tags = el.get("tags", {})
            feats.append({"type": "Feature",
                          "properties": {"highway": tags.get("highway"),
                                         "oneway": tags.get("oneway", "no"),
                                         "name": tags.get("name")},
                          "geometry": {"type": "LineString", "coordinates": coords}})
        if feats:
            json.dump({"type": "FeatureCollection", "features": feats}, open(path, "w"))
            return len(feats), url
    return 0, None


def synthetic_grid_geojson(path, n=8, spacing=0.004, lon0=-121.745, lat0=38.543):
    feats = []
    for i in range(n):
        for j in range(n):
            lon, lat = lon0 + j * spacing, lat0 + i * spacing
            if j < n - 1:
                feats.append({"type": "Feature",
                              "properties": {"highway": "residential", "oneway": "no"},
                              "geometry": {"type": "LineString",
                                           "coordinates": [[lon, lat], [lon + spacing, lat]]}})
            if i < n - 1:
                feats.append({"type": "Feature",
                              "properties": {"highway": "residential", "oneway": "no"},
                              "geometry": {"type": "LineString",
                                           "coordinates": [[lon, lat], [lon, lat + spacing]]}})
    json.dump({"type": "FeatureCollection", "features": feats}, open(path, "w"))
    return len(feats)


NET_PATH = os.path.join(WORK, "network.geojson")
n_feats, src = download_network_geojson(BBOX, NET_PATH)
if n_feats == 0:
    n_feats = synthetic_grid_geojson(NET_PATH)
    src = "synthetic grid (Overpass unreachable)"

net = rt.Network.from_geojson(NET_PATH)
print(f"{n_feats} ways from {src}")
print(f"network: {net.number_of_nodes()} nodes, {net.number_of_edges()} edges, "
      f"metric EPSG:{net.crs_metric.to_epsg()}")

## 2. Simulate GPS trajectories — timestamps + ids, **no speed**

To demonstrate speed derivation we generate vehicles that drive real routes on the
downloaded network. Each gets a unique id and is sampled every ~25 m with a
timestamp set by an hour-of-day speed profile (slow at rush hours). About a
quarter of trips include a multi-minute **parked dwell** so we can later see the
dwell-aware cleaner remove it. Crucially, the output CSV has **no speed column**.

In [ ]:
def simulate_trajectories(net, out_csv, n_trips=120, spacing_m=25.0, seed=7):
    rng = np.random.default_rng(seed)
    G = net.graph
    nodes = list(G.nodes())
    inv = net._transformer_inv
    rows, trip, attempts = [], 0, 0
    while trip < n_trips and attempts < n_trips * 30:
        attempts += 1
        cur = nodes[rng.integers(len(nodes))]
        path_edges = []
        for _ in range(int(rng.integers(6, 16))):
            succ = list(G.successors(cur))
            if not succ:
                break
            nxt = succ[int(rng.integers(len(succ)))]
            path_edges.append((cur, nxt))
            cur = nxt
        if len(path_edges) < 3:
            continue
        coords = []
        for u, v in path_edges:
            g = G.get_edge_data(u, v).get("geometry")
            if g is None:
                continue
            c = list(g.coords)
            if coords and coords[-1] == c[0]:
                c = c[1:]
            coords.extend(c)
        if len(coords) < 2:
            continue
        line = LineString(coords)
        L = line.length
        if L < spacing_m * 4:
            continue
        hour = int(rng.integers(0, 24))
        base_mph = 12.0 if hour in (7, 8, 9, 16, 17, 18) else (32.0 if hour < 6 else 22.0)
        base_mps = base_mph * 0.44704
        t0 = pd.Timestamp("2024-06-03") + pd.Timedelta(hours=hour, minutes=int(rng.integers(0, 60)))
        tid = f"veh_{trip:04d}"
        d, t = 0.0, 0.0
        dwell_at = rng.random() < 0.25
        dwell_d = L * rng.uniform(0.3, 0.7) if dwell_at else -1.0
        dwelled = False
        while d <= L:
            p = line.interpolate(d)
            lon, lat = inv.transform(p.x, p.y)
            lon += rng.normal(0, 8e-6); lat += rng.normal(0, 8e-6)  # ~1 m GPS noise
            ts = t0 + pd.Timedelta(seconds=int(round(t)))
            rows.append({"vehicle_id": tid, "longitude": float(lon),
                         "latitude": float(lat), "timestamp": ts.isoformat()})
            if dwell_at and not dwelled and d >= dwell_d:      # inject a parked dwell
                dwelled = True
                for k in range(1, 8):
                    ts2 = ts + pd.Timedelta(seconds=30 * k)
                    rows.append({"vehicle_id": tid,
                                 "longitude": float(lon + rng.normal(0, 3e-6)),
                                 "latitude": float(lat + rng.normal(0, 3e-6)),
                                 "timestamp": ts2.isoformat()})
                t += 30 * 7
            speed = max(1.0, base_mps + rng.normal(0, 1.0))
            t += spacing_m / speed
            d += spacing_m
        trip += 1
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    return trip, len(rows)


PTS_PATH = os.path.join(WORK, "gps_no_speed.csv")
ntrips, npts = simulate_trajectories(net, PTS_PATH)
raw = pd.read_csv(PTS_PATH)
print(f"{ntrips} trips, {npts} points; columns = {list(raw.columns)}  (no speed!)")
raw.head()

## 3. Derive speed from positions, then save

`load_points(derive_speed=True)` reconstructs speed per vehicle from the geodesic
distance between consecutive points over their time gap (see *statistics §7*). It
needs the `id_col` so speeds are never differenced across two vehicles. We then
persist the enriched dataset with `save_points` — now it has a speed column.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a few single-point/duplicate-time rows drop
    pts = rt.load_points(PTS_PATH, derive_speed=True, id_col="vehicle_id")

DERIVED_PATH = os.path.join(WORK, "gps_with_speed.csv")
rt.save_points(pts, DERIVED_PATH, speed_unit="mph")

mph = rt.from_mps(pts.df["speed_mps"].values, "mph")
print(f"derived speed for {len(pts)} points; median = {np.nanmedian(mph):.1f} mph")
print("saved columns:", list(pd.read_csv(DERIVED_PATH).columns))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(mph[np.isfinite(mph)], bins=40, color="#4c78a8")
ax.set_xlabel("derived speed (mph)"); ax.set_ylabel("points")
ax.set_title("Speed reconstructed from GPS positions")
plt.show()

## 4. Match to roads, then clean *dwell-aware*

`HMMMatcher` snaps each trajectory to the most probable edge sequence. For
trajectories we clean with `filter_trajectory_speed`, which removes parked
**dwells** and missing-speed points but **keeps slow-but-moving congestion** — so
we don't bias segment speeds upward by deleting the very traffic we're studying
(*statistics §8*).

In [ ]:
near = rt.NearestMatcher(net, max_dist=40).match(pts)
hmm = rt.HMMMatcher(net, sigma_z=8, beta=30, max_dist=40).match(pts)
print(f"matched to an edge — nearest: {(near['edge_id']!=-1).mean():.0%}, "
      f"HMM: {(hmm['edge_id']!=-1).mean():.0%}")

clean = rt.filter_trajectory_speed(hmm, dwell_radius_m=25, dwell_min_s=120,
                                   max_speed=80, unit="mph")
print(f"trajectory-aware cleaning: {len(hmm)} -> {len(clean)} observations "
      f"({len(hmm)-len(clean)} parked/missing removed; congestion kept)")

## 5. Aggregate by hour and detect peak / off-peak blocks

`classify_hours` splits the day into two data-driven blocks: hours at or below the
median network-wide hourly speed are *peak* (busier), the rest *off-peak*.

In [ ]:
agg = rt.aggregate_speeds(clean, block_hours=1, statistic="median",
                          output_unit="mph", min_samples=1)
cls = rt.classify_hours(clean, statistic="median")
print("peak hours:", cls["peak_hours"])
print("off-peak  :", cls["offpeak_hours"])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(agg["block_start_hour"], agg["median_speed"], color="#4c78a8", label="median mph")
for h in cls["peak_hours"]:
    ax.axvspan(h - 0.5, h + 0.5, color="red", alpha=0.10)
ax.axhline(rt.from_mps(cls["threshold_speed_mps"], "mph"), color="k", ls="--",
           lw=1, label="peak/off-peak split")
ax.set_xlabel("hour of day"); ax.set_ylabel("median speed (mph)")
ax.set_title("Network-wide hourly speed (peak hours shaded)")
ax.legend()
plt.show()

## 6. Three speeds per segment

`assign_segment_speeds` writes, for every edge, a representative speed over the
whole timeframe, the peak block, and the off-peak block (plus matching travel
times). Pooling into two broad blocks keeps far more observations per segment than
24 hourly slices — which is what makes time routing stable on sparse GPS.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    info = rt.assign_segment_speeds(net, clean, statistic="median")
cov = info["coverage"]
print(f"edges with data — overall {cov['overall']}, peak {cov['peak']}, "
      f"off-peak {cov['offpeak']}  of {info['n_edges_total']} total")

# peek at a few edges where all three regimes are observed
sample = []
for _u, _v, d in net.graph.edges(data=True):
    if all(d.get(f"obs_speed_mps_{r}") for r in ("overall", "peak", "offpeak")):
        sample.append((d["edge_id"],
                       round(rt.from_mps(d["obs_speed_mps_overall"], "mph"), 1),
                       round(rt.from_mps(d["obs_speed_mps_peak"], "mph"), 1),
                       round(rt.from_mps(d["obs_speed_mps_offpeak"], "mph"), 1)))
pd.DataFrame(sample, columns=["edge_id", "overall_mph", "peak_mph", "offpeak_mph"]).head(8)

## 7. Route by time on each regime — with real error handling

We route across the largest strongly-connected component (so a path exists) and
compare travel time in each regime. Then we deliberately trigger the no-path case
to show the actionable error instead of an opaque graph exception.

In [ ]:
import networkx as nx

router = rt.Router(net, default_speed_mps=10.0)

scc = list(max(nx.strongly_connected_components(net.graph), key=len))
xs = np.array([p[0] for p in scc]); ys = np.array([p[1] for p in scc])
o = scc[int(np.argmin(xs + ys))]            # SW-most node
d = scc[int(np.argmax(xs + ys))]            # NE-most node

print(f"routing across the largest SCC ({len(scc)} nodes):")
for per in ("overall", "peak", "offpeak"):
    r = router.route(o, d, mode="time", period=per)
    print(f"  {per:>7}: {r['travel_time_s']:5.0f} s over {r['distance_m']:.0f} m "
          f"({r['n_edges']} edges, {r['n_edges_default']} on default speed)")

# actionable error on an unreachable destination
net.graph.add_edge((-200.0, 0.0), (-200.001, 0.0), edge_id=10**9, length_m=90.0, geometry=None)
try:
    rt.Router(net).route((-200.0, 0.0), o, mode="distance")
except ValueError as e:
    print("\nno-path handled:", e)
net.graph.remove_node((-200.0, 0.0)); net.graph.remove_node((-200.001, 0.0))

## 8. Map it — segment peak speeds + a route

An interactive `folium` map: each road segment coloured by its **peak-hour**
speed (red = slow/busy, green = fast), with the time-optimal peak route overlaid
in blue.

In [ ]:
import folium
import matplotlib
from matplotlib.colors import Normalize, to_hex

inv = net._transformer_inv
peak_speeds = [d.get("obs_speed_mps_peak") for _, _, d in net.graph.edges(data=True)
               if d.get("obs_speed_mps_peak")]
vmin = rt.from_mps(min(peak_speeds), "mph"); vmax = rt.from_mps(max(peak_speeds), "mph")
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = matplotlib.colormaps["RdYlGn"]

allnodes = list(net.graph.nodes())
m = folium.Map(location=[np.mean([n[1] for n in allnodes]),
                         np.mean([n[0] for n in allnodes])],
               zoom_start=15, tiles="cartodbpositron")

for _u, _v, dd in net.graph.edges(data=True):
    g = dd.get("geometry")
    if g is None:
        continue
    lon, lat = inv.transform(np.asarray(g.coords)[:, 0], np.asarray(g.coords)[:, 1])
    sp = dd.get("obs_speed_mps_peak")
    color = to_hex(cmap(norm(rt.from_mps(sp, "mph")))) if sp else "#bbbbbb"
    folium.PolyLine(list(zip(lat, lon)), color=color, weight=4, opacity=0.85,
                    tooltip=(f"{rt.from_mps(sp,'mph'):.0f} mph peak" if sp else "no data")
                    ).add_to(m)

rr = router.route(o, d, mode="time", period="peak")
route_ll = router.route_geometry_lonlat(rr)
folium.PolyLine([(la, lo) for lo, la in route_ll], color="blue", weight=6,
                opacity=0.9, tooltip="peak time-optimal route").add_to(m)
folium.Marker([o[1], o[0]], tooltip="origin").add_to(m)
folium.Marker([d[1], d[0]], tooltip="destination").add_to(m)
m

## Wrap-up

You loaded a real OSM network, **derived speed from raw GPS positions**, cleaned
trajectories without throwing away congestion, summarised each segment over
overall / peak / off-peak blocks, and routed and mapped the result — all on the
lightweight `roadtraffic` core.

To run on your own data: drop the simulation (cell 2) and point `load_points` at
your GPS file (`derive_speed=True` only if it lacks a speed column), and set
`BBOX` to your study area. See `docs/statistics.md` for the methodology behind
every number.